In [1]:
import pandas as pd
import protfasta
import glob

# Running mmseqs2
Generating commands (run from mmseqs folder of output)

In [3]:
# This will give you the commands to run MMseqs for your sequences
def db_name(prefix, min_seq_id=0.5, fasta_path=False):
    if fasta_path:
        print(f"mmseqs createdb {fasta_path} {prefix}_DB") # - this alr exists
    id_thresh = int(min_seq_id * 100)
    print(f"mmseqs cluster {prefix}_DB {prefix}_DB_clu_id{id_thresh} {prefix}tmp_id{id_thresh} --min-seq-id {min_seq_id}")                      
    print(f"mmseqs createtsv {prefix}_DB {prefix}_DB {prefix}_DB_clu_id{id_thresh} {prefix}_DB_clu_id{id_thresh}.tsv")


In [4]:
# This prints the commands used to run mmseqs
db_name("CL_SK_plant_seqs_disord_thresh", 0.75, "data/CL_SK_plant_seqs_disord_thresh.fasta")

mmseqs createdb ../../data/CL_SK_plant_seqs_disord_thresh.fasta CL_SK_plant_seqs_disord_thresh_DB
mmseqs cluster CL_SK_plant_seqs_disord_thresh_DB CL_SK_plant_seqs_disord_thresh_DB_clu_id75 CL_SK_plant_seqs_disord_threshtmp_id75 --min-seq-id 0.75
mmseqs createtsv CL_SK_plant_seqs_disord_thresh_DB CL_SK_plant_seqs_disord_thresh_DB CL_SK_plant_seqs_disord_thresh_DB_clu_id75 CL_SK_plant_seqs_disord_thresh_DB_clu_id75.tsv



In [16]:
# Read in the clusters
CL_SK_plant_seqs_disord_thresh_DB_clu_id75 = pd.read_csv("data/CL_SK_plant_seqs_disord_thresh_DB_clu_id75.tsv", sep = "\t", header = None)

# These are the roughly 240k Sequences I want to train the consensus model on!
rep_seqs = pd.DataFrame(CL_SK_plant_seqs_disord_thresh_DB_clu_id75[0].value_counts()).reset_index()
rep_seqs

,0,count
0,A0A452V6A1,544
1,A0A7K8ZQK0,499
2,H3BAF8,426
3,g000739.m1_yHMPu5000034983_candida_tsuchiyae_2...,344
4,A0A6P5JAN9,300
...,...,...
243146,Zosma10g00510.1,1
243147,Zosma287g00130.1,1
243148,Zosma20g01410.1,1
243149,Zosma115g00200.1,1


In [6]:
CL_SK_plant_seqs_disord_thresh_DB_clu_id75[0]

0                                                    Q8GGH0
1                                                    F1R2J1
2                                                    O75364
3                                                    O75364
4                                                    O75364
                                ...                        
497978    g002397.m1_yHMPu5000041691_saccharomycopsis_cr...
497979    g002397.m1_yHMPu5000041691_saccharomycopsis_cr...
497980      g002544.m1_yHDO593_barnettozyma_sp_180604.final
497981    g003106.m1_yHMPu5000034601_starmera_dryadoides...
497982    g002629.m1_yHMPu5000034636_ogataea_nitratoaver...
Name: 0, Length: 497983, dtype: object

In [10]:
cl_seqs = protfasta.read_fasta("data/cl_seqs.fasta", invalid_sequence_action='remove')
cl_seqs_df = pd.DataFrame({"id": cl_seqs.keys(), "seq": cl_seqs.values()})
cl_seqs_df.head()

,id,seq
0,g002981.m1_ashbya_aceri.final,MNRSDELQDAEDSGQQRYETYSCSRCRRLKKRCSKELPRCFSCEAA...
1,g001536.m1_brettanomyces_anomalus.final,MSDKPHNQYACTRCRRLKKKCSKEFPRCANCRKQNKVCEYVERKNK...
2,C1_01200W_A_candida_albicans.cgd,MRPIDSIVSKFRVQKHTSNLADNPAVLKHPEDMPKLTSCLRCRKLK...
3,g000106.m1_candida_albicans.final,MRPIDSIVSKFRVQKHTSNLADNPAVLKHPEDMPKLTSCLRCRKLK...
4,g001050.m1_candida_dubliniensis.final,MRPTDSIVSKFRVQKPTSNLANNPAVLRHPENMPKLTSCLRCRKLK...


In [21]:
sk_seqs = protfasta.read_fasta("data/interpro_uniprot_evid_1_2.fasta", invalid_sequence_action='ignore')
sk_seqs_df = pd.DataFrame({"id": sk_seqs.keys(), "seq": sk_seqs.values()})
sk_seqs_df["id"] = sk_seqs_df["id"].str.split("|").str[1]
sk_seqs_df.head()

,id,seq
0,A0A0B4K6B8,MADDEQFSLCWNNFNTNLSAGFHESLCRGDLVDVSLAAEGQIVKAH...
1,A0A0B4K7A6,MADDEQFSLCWNNFNTNLSAGFHESLCRGDLVDVSLAAEGQIVKAH...
2,A0A5S9MMK5,MMTTTVQKNCWRLDQTMLGLEKPGSSDISSSSTDTSAISPISVSSM...
3,Q4VC44,MPLPEPSEQEGESVKAGQEPSPKPGTDVIPAAPRKPREFSKLVLLT...
4,Q86B87,MADDEQFSLCWNNFNTNLSAGFHESLCRGDLVDVSLAAEGQIVKAH...


In [25]:
plant_seqs = protfasta.read_fasta("data/PlantTFDB-all_TF_pep.fas", invalid_sequence_action='ignore')
plant_seqs_df = pd.DataFrame({"id": plant_seqs.keys(), "seq": plant_seqs.values()})
plant_seqs_df["id"] = plant_seqs_df["id"].str.split(" ").str[0]
plant_seqs_df.head()

,id,seq
0,KFK36254.1,MYSAIRSLPVDGSMGEYSDGTNLPIEACLVLTTDPKPRLRWTTELH...
1,KFK25038.1,MAPKHTDTIQNPPMPSSSSEEESASSGEDSDSSKKHNSASDSDPIK...
2,KFK35037.1,MTTTTTPHWPDFSSQKLPSIAATAAATAATAGQNQQNQNPSWMDEF...
3,KFK44643.1,MEGERHDSTTSSSRKRSHEPAEERPASHVATMAFKEKPYRGIRMRK...
4,KFK32619.1,MAKPPQFFHTLVNTHLVIPKDFFSKYIQGKSVDDTVELKSDSTDIN...


In [26]:
rep_seqs["SK?"] = rep_seqs[0].isin(sk_seqs_df["id"])
rep_seqs["CL?"] = rep_seqs[0].isin(cl_seqs_df["id"])
rep_seqs["plant?"] = rep_seqs[0].isin(plant_seqs_df["id"])
rep_seqs

,0,count,SK?,CL?,plant?
0,A0A452V6A1,544,True,False,False
1,A0A7K8ZQK0,499,True,False,False
2,H3BAF8,426,True,False,False
3,g000739.m1_yHMPu5000034983_candida_tsuchiyae_2...,344,False,True,False
4,A0A6P5JAN9,300,True,False,False
...,...,...,...,...,...
243146,Zosma10g00510.1,1,False,False,True
243147,Zosma287g00130.1,1,False,False,True
243148,Zosma20g01410.1,1,False,False,True
243149,Zosma115g00200.1,1,False,False,True


In [27]:
rep_seqs.sum(numeric_only=True)

count     497983
SK?        38192
CL?        98214
plant?    106745
dtype: int64